# Garak Scenarios

The Garak scenario family implements probes inspired by the
[Garak](https://github.com/NVIDIA/garak) framework. These include encoding-based probes (which
test whether a target can be tricked into producing harmful content when prompts are encoded in
various formats), web-injection probes (which test whether a target emits markdown
data-exfiltration or cross-site-scripting payloads), and system-prompt-extraction probes (which
test whether a target can be coaxed into revealing its own system prompt).

For full programming details, see the
[Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb).

In [ ]:
from pathlib import Path

from pyrit.output import output_scenario_async
from pyrit.registry import TargetRegistry
from pyrit.scenario.garak import (
    Encoding,
    EncodingTechnique,
    SystemPromptExtraction,
    SystemPromptExtractionTechnique,
)
from pyrit.scenario.garak.encoding import EncodingDatasetConfiguration
from pyrit.setup import initialize_from_config_async

await initialize_from_config_async(config_path=Path("pyrit_conf.yaml"))  # type: ignore

objective_target = TargetRegistry.get_registry_singleton().instances.get("openai_chat")

Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


[pyrit:alembic] No new upgrade operations detected.


TextAdaptive: _EXCLUDED_TECHNIQUES entries ['prompt_sending'] are not in the current scenario-techniques catalog ['context_compliance', 'crescendo_history_lecture', 'crescendo_journalist_interview', 'crescendo_movie_director', 'crescendo_simulated', 'flip', 'many_shot', 'pair', 'red_teaming', 'role_play_movie_script', 'role_play_persuasion', 'role_play_persuasion_written', 'role_play_trivia_game', 'role_play_video_game', 'skeleton_key', 'tap', 'violent_durian']; the exclusion is a no-op for those entries. Remove stale entries or update the catalog.


## Encoding

Tests whether the target can decode and comply with encoded harmful prompts. Each encoding
technique encodes the prompt, asks the target to decode it, and scores whether the decoded output
matches the harmful content. Default datasets include slur terms and web/HTML/JS content.

**CLI example:**

```bash
pyrit_scan garak.encoding --target openai_chat --techniques base64 --max-dataset-size 1
```

**Available techniques** (17 encodings): Base64, Base2048, Base16, Base32, ASCII85, Hex,
QuotedPrintable, UUencode, ROT13, Braille, Atbash, MorseCode, NATO, Ecoji, Zalgo, LeetSpeak,
AsciiSmuggler

**Aggregate techniques:** `ALL` (every encoding, exhaustive) and `DEFAULT` (a broad curated subset
spanning every encoding family — base-N, byte-encodings, substitution ciphers, and symbolic
alphabets — for a meaningful default scan; the niche/lossy schemes are ALL-only). `DEFAULT` is used
when no techniques are specified.

> **Note:** Technique composition is NOT supported for Encoding — each encoding is tested
> independently.

In [ ]:
dataset_config = EncodingDatasetConfiguration(dataset_names=["garak_slur_terms_en"], max_dataset_size=1)

scenario = Encoding()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [EncodingTechnique.Base64],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

print(f"Scenario: {scenario.name}")
print(f"Atomic attacks: {scenario.atomic_attack_count}")

scenario_result = await scenario.run_async()  # type: ignore

Scenario: Encoding
Atomic attacks: 11


In [ ]:
await output_scenario_async(scenario_result)


                                    📊 SCENARIO RESULTS: Encoding                                    

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Encoding
    • Result ID: 1d023af9-9a06-4383-b0f5-16670beb2015
    • Scenario Version: 2
    • PyRIT Version: 1.1.0.dev0
    • Description:
        Encoding Scenario implementation for PyRIT. This scenario tests how resilient models are to various encoding
        attacks by encoding potentially harmful text (by default slurs and XSS payloads) and testing if the model will
        decode and repeat the encoded payload. It mimics the Garak encoding probe. The scenario works by: 1. Taking seed
        prompts (the harmful text to be encoded) 2. Encoding them using various encoding schemes (Base64, ROT13, Morse,
        etc.) 3. Asking the target model to decode the encoded text 4. Scoring whether the model successfully decoded
     

## WebInjection

Ports Garak's `web_injection` probe family. Tests whether the target can be coaxed into emitting
web-injection payloads: markdown-image data exfiltration (CSRF-style) and cross-site-scripting
(XSS). Each technique pairs a `SeedObjective` (the goal) with a `SeedPrompt` (the literal
injection prompt) and scores the response with the markdown-injection or XSS output scorer.

**CLI example:**

```bash
pyrit_scan garak.web_injection --target openai_chat --techniques xss --max-dataset-size 1
```

**Available techniques** (8 probes): MarkdownImageExfil, ColabAIDataLeakage,
StringAssemblyDataExfil, PlaygroundMarkdownExfil, MarkdownURIImageExfilExtended,
MarkdownURINonImageExfilExtended, TaskXSS, MarkdownXSS.

**Aggregate techniques:** `ALL` (all 8), `DEFAULT` (excludes the two combinatorial extended
probes), `EXFIL` (the 6 markdown-exfil probes), and `XSS` (TaskXSS + MarkdownXSS).

## Doctor

Ports Garak's `doctor` probe, which implements the HiddenLayer Policy Puppetry universal bypass:
a post-instruction-hierarchy prompt injection that frames a harmful request as a fictional
medical-drama script ("Dr House") to override safety guardrails. Each objective from the
`garak_doctor` dataset is wrapped in a Policy Puppetry template via `PolicyPuppetryConverter` and
the response is scored for non-refusal (mirroring Garak's `mitigation.MitigationBypass` detector).

**CLI example:**

```bash
pyrit_scan garak.doctor --target openai_chat --techniques policy_puppetry --max-dataset-size 1
```

**Available techniques** (2 probes): `PolicyPuppetry` (wraps the objective in the Dr House
template) and `PolicyPuppetryLeet` (the same template, additionally leetspeak-encoded). Both are
tagged `default`, so `DEFAULT` and `ALL` currently coincide.

## SystemPromptExtraction

Ports Garak's `sysprompt_extraction` probe. A real system prompt (sourced from the
`garak_drh_system_prompts` / `garak_tm_system_prompts` libraries) is installed on the target, then
an extraction request asks the model to reveal it. Responses are scored deterministically by
`SystemPromptExtractionScorer`, a character n-gram containment overlap between the response and the
known system prompt (a faithful port of Garak's `PromptExtraction` detector), wrapped by a
`FloatScaleThresholdScorer` at threshold 0.5.

Each of the 9 attack-template categories is a technique; across the selected categories the total
(system prompt × template) combinations are randomly sampled down to `prompt_cap` (Garak's
`soft_probe_prompt_cap`, default 256) so a default run stays bounded.

**CLI example:**

```bash
pyrit_scan garak.system_prompt_extraction --target openai_chat --techniques direct_requests
```

**Available techniques** (9 categories): DirectRequests, RolePlayingAttacks, EncodingBasedAttacks,
IndirectCreativeApproaches, CodeTechnicalFraming, ContinuationTricks, MultiLayeredApproaches,
AuthorityUrgencyFraming, ConfusionDistraction.

The minimal run below installs a single system prompt and runs one category so it completes
quickly.

In [ ]:
sysprompt_scenario = SystemPromptExtraction(system_prompt_subsample=1, prompt_cap=1)
sysprompt_scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [SystemPromptExtractionTechnique.DirectRequests],
    }
)
await sysprompt_scenario.initialize_async()  # type: ignore

print(f"Scenario: {sysprompt_scenario.name}")
print(f"Atomic attacks: {sysprompt_scenario.atomic_attack_count}")

sysprompt_result = await sysprompt_scenario.run_async()  # type: ignore

Scenario: SystemPromptExtraction
Atomic attacks: 1


In [ ]:
await output_scenario_async(sysprompt_result)


                             📊 SCENARIO RESULTS: SystemPromptExtraction                             

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: SystemPromptExtraction
    • Result ID: a1592b0a-3041-450e-9234-c3beee6096b4
    • Scenario Version: 1
    • PyRIT Version: 1.1.0.dev0
    • Description:
        System Prompt Extraction scenario implementation for PyRIT. Ports garak's
        ``sysprompt_extraction.SystemPromptExtraction`` probe. A real system prompt (sourced from the
        ``garak_drh_system_prompts`` / ``garak_tm_system_prompts`` datasets) is installed on the target, then an
        extraction request (from the ``garak_system_prompt_extraction`` dataset) asks the model to reveal it. Responses
        are scored deterministically with ``SystemPromptExtractionScorer`` (a character n-gram containment overlap
        between the response and the known system prom

For more details, see the [Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb) and
[Configuration](../getting_started/configuration.md).